# CTD Profile Grapher

Interactive depth profiles from Sea-Bird `.cnv` files — no Excel step, no Google Drive.

**How to use**
1. Run **Setup** once per session.
2. Run **1 · Survey location and files** — type where you sampled, pick your `.cnv` files, then tick which variables you want graphed. The ones the notebook recognises are already ticked; anything else your files contain is listed too, in case you want it. Depth is the Y axis unless you change it.
3. Run **2 · Draw the graphs**. To look at part of the water column, fill in `depth_from_m` and `depth_to_m` and run that cell again. Your files stay loaded, so there is no need to upload twice.

**Naming:** the filename becomes the legend label, with underscores turned into spaces — `Station_1.cnv` → **Station 1**, `East_Passage.cnv` → **East Passage**. Stations sort naturally (1, 2, … 10, 11).

**Colours** are locked to station order, so the first station is always the same blue, the second always the same red, and so on. The Load cell prints the colour key so a bar chart or a station map can use exactly the same colours.

## What you get

Files land in `/content/CTD_output/` — open the folder icon in the left sidebar to download them.

- **`png/*.png`** — ordinary pictures, one per variable. Use these anywhere you would use a photo.
- **`CTD_profiles.html`** — all the graphs together, interactive, around 30 KB. Hovering shows exact values; you can zoom, or hide a station by clicking it in the legend. Needs internet to open.
- **`single_graphs/*.html`** — the same graphs but **one file each**: `Temperature.html`, `Salinity.html`, and so on. Take just the one you want.
- **`CTD_profiles_offline.html`** — all the graphs with everything built in, several MB, works with no connection at all. Only worth taking if you will be presenting somewhere without wifi.

## Sharing a graph

**For a document or slideshow — use the PNG.** Google Docs, Word, Google Slides and PowerPoint cannot display an interactive chart, and no setting or add-on changes that. Insert the picture like any other image.

**To let someone explore it — send them `CTD_profiles.html`.** They double-click it and it opens in their browser with everything working, nothing to install. This is the easy answer and it covers almost every case.

**Want just one graph, not all of them?** Use the matching file from `single_graphs/` — `Temperature.html` is the depth vs temperature graph on its own, and nothing else. Treat it exactly like the file above: send it, or put it online for a link. Each one is about 15 KB.

**To put a graph inside a web page**, hand the HTML file to whoever manages the site and ask them to embed it in an iframe. That is a normal request and they will know what it means. Google Drive will not work as a substitute — it downloads the file rather than displaying it.

The sensor set is read from each file's own header, so casts from different instruments work with no setting to change.

## Credit

Example data collected by students of **TGEOS 445, Estuarine Field Studies, University of Washington Tacoma**, Spring 2026, in Colvos Passage and East Passage, Puget Sound.

Instrument: **Sea-Bird SBE 19plus** (temperature and conductivity SN 7686), processed with Sea-Bird SBEDataProcessing.

In [ ]:
#@title Setup — run once per session
!pip install -q "kaleido==0.2.1" 2>/dev/null

import os, re, colorsys
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# ─────────────── settings ───────────────
LINE_SHAPE   = "spline"              # "spline" (smooth) or "linear" (raw bins)
SHOW_MARKERS = False                 # True → dot at each 1-db bin
LINE_WIDTH   = 1.5
X_PAD_FRAC   = 0.05                  # breathing room at left/right edges (5%)
Y_PAD_FRAC   = 0.02
AXIS_FONT    = 13                    # x and y axis titles share this size
EXPORT_PNG   = True
OUT_DIR      = "/content/CTD_output"
LOCATION     = ""                    # set by the Load cell, e.g. "Quartermaster Harbor"
# ────────────────────────────────────────

# Canonical variable → the Sea-Bird short names that carry it, in priority order,
# each with its own unit. Different instruments report the same quantity under
# different names AND different units — oxygen as mL/L on one cast and µmol/kg on
# the next — so the unit travels with the name rather than with the variable.
# Whatever a file actually contains is what gets plotted.
VARIABLES = [
    ("Temperature", [("tv290c", "°C"), ("t090c", "°C"), ("t190c", "°C"),
                     ("tv190c", "°C"), ("t068c", "°C"), ("t168c", "°C"),
                     ("t090", "°C"), ("t190", "°C")]),
    ("Salinity", [("sal00", "PSU"), ("sal11", "PSU")]),
    ("Density (sigma-t)", [("sigma-t00", "kg/m³"), ("sigma-t11", "kg/m³"),
                           ("sigma-e00", "kg/m³"), ("sigma-é00", "kg/m³")]),
    ("Dissolved Oxygen", [("sbeox0ml/l", "mL/L"), ("sbeox1ml/l", "mL/L"),
                          ("oxml/l", "mL/L"),
                          ("sbeox0mm/kg", "µmol/kg"), ("sbeox1mm/kg", "µmol/kg"),
                          ("sbeox0mg/l", "mg/L"), ("sbeox1mg/l", "mg/L"),
                          ("sbeox0ps", "% sat"), ("sbeox1ps", "% sat")]),
    ("Fluorescence", [("fleco-afl", "mg/m³"), ("flecoafl", "mg/m³"),
                      ("flcuva", "mg/m³"), ("flsp", "mg/m³"),
                      ("flc", "mg/m³"), ("wetstar", "mg/m³")]),
    ("Beam Transmission", [("cstartr0", "%"), ("cstartr1", "%"), ("xmiss", "%")]),
    ("Turbidity", [("turbwetntu0", "NTU"), ("turbwetntu1", "NTU"),
                   ("obs", "NTU"), ("seaturbmtr", "NTU")]),
    ("pH", [("ph", "")]),
    ("PAR", [("par", "µmol photons/m²/s")]),
    ("CDOM", [("wetcdom", "mg/m³")]),
]

# Vertical axis. Depth is preferred; pressure stands in when a file has no depth
# channel, and is labelled as pressure rather than quietly called metres.
DEPTH_CANDIDATES = [("depsm", "Depth (m)"), ("depfm", "Depth (m)"),
                    ("prdm", "Pressure (db)"), ("prsm", "Pressure (db)"),
                    ("pr", "Pressure (db)")]

# Fixed station palette. Position decides colour: the 1st station in the
# canonical order is always PALETTE[0], the 2nd always PALETTE[1], and so on.
# Reuse STATION_COLORS in a bar chart or a station map and the colours agree
# across every figure you make.
PALETTE = ["#1f77b4", "#d62728", "#2ca02c", "#ff7f0e", "#9467bd",
           "#8c564b", "#e377c2", "#17becf", "#bcbd22", "#7f7f7f"]

STATION_COLORS = {}
_EXTRA_COLORS = []          # generated colours beyond PALETTE, in order
_LAB = {}


def _to_lab(hexcolor):
    """sRGB hex → CIE Lab, so colours can be compared the way an eye does.
    Plain RGB distance calls greens near-identical that clearly are not."""
    if hexcolor in _LAB:
        return _LAB[hexcolor]
    r, g, b = (int(hexcolor[i:i + 2], 16) / 255 for i in (1, 3, 5))
    inv = lambda c: c / 12.92 if c <= 0.04045 else ((c + 0.055) / 1.055) ** 2.4
    r, g, b = inv(r), inv(g), inv(b)
    x = (0.4124 * r + 0.3576 * g + 0.1805 * b) / 0.95047
    y = 0.2126 * r + 0.7152 * g + 0.0722 * b
    z = (0.0193 * r + 0.1192 * g + 0.9505 * b) / 1.08883
    f = lambda t: t ** (1 / 3) if t > 0.008856 else 7.787 * t + 16 / 116
    x, y, z = f(x), f(y), f(z)
    _LAB[hexcolor] = (116 * y - 16, 500 * (x - y), 200 * (y - z))
    return _LAB[hexcolor]


def _candidate_colors():
    out = []
    for h in range(36):
        for light in (0.36, 0.50, 0.64):
            for sat in (0.50, 0.75, 0.95):
                r, g, b = colorsys.hls_to_rgb(h / 36, light, sat)
                out.append("#{:02x}{:02x}{:02x}".format(
                    round(r * 255), round(g * 255), round(b * 255)))
    return out


def _ensure_colors(n):
    """Grow the colour list to n entries, each new colour chosen as the one
    furthest from every colour already in use. Greedy farthest-point picking
    keeps large sets readable where evenly-spaced hues do not."""
    if len(PALETTE) + len(_EXTRA_COLORS) >= n:
        return
    cands = _candidate_colors()
    used = [_to_lab(c) for c in PALETTE + _EXTRA_COLORS]
    while len(PALETTE) + len(_EXTRA_COLORS) < n:
        best, best_d = None, -1.0
        for c in cands:
            lc = _to_lab(c)
            d = min((lc[0] - u[0]) ** 2 + (lc[1] - u[1]) ** 2 + (lc[2] - u[2]) ** 2
                    for u in used)
            if d > best_d:
                best_d, best = d, c
        _EXTRA_COLORS.append(best)
        used.append(_to_lab(best))


def station_color(i):
    """Colour for the i-th station. Lines are always solid, so past the base
    palette new colours are generated rather than repeated. Deterministic:
    station i gets the same colour every run, for any n."""
    if i < len(PALETTE):
        return PALETTE[i]
    _ensure_colors(i + 1)
    return _EXTRA_COLORS[i - len(PALETTE)]


def assign_station_styles(labels):
    """Lock each station to a colour by its position in the canonical order.

    Call once after loading. The returned dict is the colour key for this
    survey — reuse it in any other chart of the same stations."""
    STATION_COLORS.clear()
    for i, lab in enumerate(labels):
        STATION_COLORS[lab] = station_color(i)
    return STATION_COLORS


def natkey(s):
    """Sort so Station 2 comes before Station 10."""
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", str(s))]


# Colab's uploader never overwrites: upload Station_1.cnv twice and the second
# copy lands as "Station_1 (1).cnv", counting up on each repeat.
DUP_SUFFIX = re.compile(r"\s*\((\d+)\)\s*$")


def station_label(filename):
    """Station_1.cnv -> 'Station 1';  East_Passage.cnv -> 'East Passage'.
    A trailing ' (1)' left by a repeated upload is dropped."""
    base = os.path.splitext(os.path.basename(filename))[0]
    return DUP_SUFFIX.sub("", base).replace("_", " ").strip()


def upload_generation(filename):
    """How many times this file has been re-uploaded. Colab counts upward, so
    the highest number is the most recent copy."""
    base = os.path.splitext(os.path.basename(filename))[0]
    m = DUP_SUFFIX.search(base)
    return int(m.group(1)) if m else 0


def dedupe_uploads(raw):
    """Keep only the newest copy of each station.

    Re-uploading to fix a mistake would otherwise plot the station twice — once
    from the bad file and once from the good one — and shift every station's
    colour, since colour follows position."""
    best = {}
    for fn in raw:
        lab, gen = station_label(fn), upload_generation(fn)
        if lab not in best or gen > best[lab][0]:
            best[lab] = (gen, fn)
    keep = {fn for _, fn in best.values()}
    for fn in raw:
        if fn not in keep:
            print(f"  IGNORED  {fn} — superseded by a newer upload of "
                  f"'{station_label(fn)}'")
    return {fn: raw[fn] for fn in raw if fn in keep}


def find_channel(df, pairs):
    """First matching (short name, unit) pair, case-insensitive.
    Returns (column, unit), or (None, None) if the file carries none of them.
    First occurrence wins when a name appears twice."""
    lower = {}
    for c in df.columns:
        lower.setdefault(str(c).lower(), c)
    for name, unit in pairs:
        if name in lower:
            return lower[name], unit
    return None, None


def downcast_only(df, depth_col):
    """Keep the downcast only, dropping scans where the package stalled or was
    pulled back up by ship heave. Returns (cleaned, rows_removed).

    This is not Sea-Bird's loopedit. That one works from raw scan-rate pressure,
    computes descent velocity against time and drops anything below a threshold.
    Working from a file alone there may be no time channel, so this takes the
    geometric route: cut at the deepest reading to remove the upcast, then keep
    only readings deeper than everything before them. The result is the clean
    monotonic downcast that loopedit exists to produce, which is what a profile
    needs to be readable — but it is an approximation, so a file that already
    says loopedit in its header is left alone."""
    if depth_col is None or len(df) < 3:
        return df, 0
    d = pd.to_numeric(df[depth_col], errors="coerce").to_numpy(dtype=float)
    if np.all(np.isnan(d)):
        return df, 0
    down = df.iloc[:int(np.nanargmax(d)) + 1]
    dd = pd.to_numeric(down[depth_col], errors="coerce").to_numpy(dtype=float)
    running = np.maximum.accumulate(np.nan_to_num(dd, nan=-np.inf))
    cleaned = down[dd >= running]
    return cleaned, len(df) - len(cleaned)


# Instrument bookkeeping rather than measurements — never offered as plottable.
HOUSEKEEPING = {"scan", "flag", "nbin", "pumps", "bpos", "pla", "timej", "times",
                "timem", "timek", "timen", "latitude", "longitude", "dz/dtm", "accm"}


def _col_by_name(df, name):
    """Exact column lookup, case-insensitive."""
    for c in df.columns:
        if str(c).lower() == str(name).lower():
            return c
    return None


def available_channels(stations):
    """What these casts can plot.

    Returns (recognised, extras, y_default). `recognised` are the variables the
    notebook knows by name, with units. `extras` is every other measured column
    the files happen to carry — offered so nothing is hidden, just unticked by
    default. `y_default` is the depth or pressure channel."""
    recognised, seen = [], set()
    for label, cands in VARIABLES:
        for _st, df in stations:
            col, unit = find_channel(df, cands)
            if col is not None:
                recognised.append({"name": label, "col": str(col),
                                   "label": f"{label} ({unit})" if unit else label})
                seen.add(str(col).lower())
                break

    ycol, ylab = None, None
    for _st, df in stations:
        c, l = find_channel(df, DEPTH_CANDIDATES)
        if c is not None:
            ycol, ylab = str(c), l
            break
    if ycol:
        seen.add(ycol.lower())

    extras = []
    for _st, df in stations:
        for c in df.columns:
            lc = str(c).lower()
            if lc in seen or lc in HOUSEKEEPING or (lc.startswith("v") and lc[1:].isdigit()):
                continue
            seen.add(lc)
            extras.append({"name": str(c), "col": str(c), "label": str(c)})

    # is_depth drives the depth window and the surface anchoring; invert only
    # flips the axis. They are separate so unticking "invert" does not quietly
    # disable the depth window as well.
    y_default = {"name": (ylab or "Depth (m)").split(" (")[0],
                 "col": ycol, "label": ylab or "Depth (m)",
                 "is_depth": True, "invert": True}
    return recognised, extras, y_default


def parse_cnv(text, source_name):
    """Parse a Sea-Bird .cnv. Columns come from the '# name' header lines and the
    data starts after *END*, so header length never has to be hardcoded."""
    lines = text.splitlines()
    col_names, bad_flag, end_idx, processing = {}, -9.99e-29, None, set()

    for i, raw in enumerate(lines):
        s = raw.strip()
        if s.upper() == "*END*":
            end_idx = i
            break
        m = re.match(r"#\s*name\s+(\d+)\s*=\s*([^:]+):", s)
        if m:
            col_names[int(m.group(1))] = m.group(2).strip()
            continue
        m = re.match(r"#\s*bad_flag\s*=\s*(\S+)", s)
        if m:
            try:
                bad_flag = float(m.group(1))
            except ValueError:
                pass
            continue
        low = s.lower()
        for tag in ("loopedit", "binavg", "wfilter", "filter", "derive",
                    "alignctd", "celltm", "split", "wildedit"):
            if low.startswith("# " + tag):
                processing.add(tag)

    if end_idx is None:
        raise ValueError(f"{source_name}: no *END* marker — is this a Sea-Bird .cnv?")
    if not col_names:
        raise ValueError(f"{source_name}: no '# name' column definitions in header.")

    ordered = [col_names[k] for k in sorted(col_names)]
    rows = []
    for raw in lines[end_idx + 1:]:
        parts = raw.split()
        if len(parts) != len(ordered):
            continue
        try:
            rows.append([float(x) for x in parts])
        except ValueError:
            continue

    df = pd.DataFrame(rows, columns=ordered)
    if not df.empty:
        # bad_flag is ~1e-29, so the comparison must be purely relative (atol=0),
        # otherwise every near-zero reading would be wiped out.
        mask = np.isclose(df.values.astype(float), bad_flag, rtol=1e-6, atol=0.0)
        df = df.mask(pd.DataFrame(mask, index=df.index, columns=df.columns))
    return df, processing


def load_files():
    """Show the upload button and read whatever is picked."""
    from google.colab import files as colab_files
    print("Select one or more .cnv files:")
    # .cnv headers are cp1252, not UTF-8 (e.g. the theta in sigma-theta)
    return dedupe_uploads(
        {n: b.decode("latin-1") for n, b in colab_files.upload().items()})


def _padded(lo, hi, frac):
    """Range with breathing room. Falls back sensibly if the series is flat."""
    span = hi - lo
    pad = span * frac if span > 0 else (abs(hi) * frac if hi else 1.0) or 1.0
    return lo - pad, hi + pad


def build_figures(stations, series, y_channel, depth_min=None, depth_max=None):
    """One figure per entry in `series`, every station overlaid.

    series      [{"name","label","col"}, ...]  x axis, one figure each
    y_channel   {"name","label","col","invert"}  shared y axis

    y_channel["is_depth"] says whether the depth window and surface anchoring
    apply; y_channel["invert"] only flips the axis direction, and is the user's
    tick box. Put a non-depth variable on y and it becomes an ordinary scatter —
    salinity against temperature, say — where no depth window makes sense."""
    if not STATION_COLORS:
        assign_station_styles([s for s, _ in stations])
    is_depth = bool(y_channel.get("is_depth", False))
    invert = bool(y_channel.get("invert", False))
    ylabel = y_channel["label"]
    figs = []

    for s in series:
        traces = []
        xlo = ylo = np.inf
        xhi = yhi = -np.inf
        for i, (st, df) in enumerate(stations):
            xcol = _col_by_name(df, s["col"])
            ycol = _col_by_name(df, y_channel["col"])
            if xcol is None or ycol is None:
                continue
            sub = df[[xcol, ycol]].dropna()
            if is_depth and depth_min is not None:
                sub = sub[sub[ycol] >= depth_min]
            if is_depth and depth_max is not None:
                sub = sub[sub[ycol] <= depth_max]
            if sub.empty:
                continue
            xlo, xhi = min(xlo, sub[xcol].min()), max(xhi, sub[xcol].max())
            ylo, yhi = min(ylo, sub[ycol].min()), max(yhi, sub[ycol].max())
            traces.append(go.Scatter(
                x=sub[xcol], y=sub[ycol], name=st,
                mode="lines+markers" if SHOW_MARKERS else "lines",
                line=dict(shape=LINE_SHAPE, width=LINE_WIDTH,
                          color=STATION_COLORS.get(st, station_color(i))),
                marker=dict(size=4),
                hovertemplate=(f"<b>{st}</b><br>{s['label']}: %{{x:.3f}}"
                               f"<br>{ylabel}: %{{y:.3f}}<extra></extra>"),
            ))
        if not traces:
            continue

        x0, x1 = _padded(xlo, xhi, X_PAD_FRAC)
        if is_depth:
            # Anchor the depth axis to the window that was ASKED for, not to the
            # outermost reading. Bins sit at bin centres (10.907 ... 19.831), so a
            # 10–20 m window holds no reading at exactly 10 or 20; anchoring to the
            # data would push both those lines off the frame. Blank means surface
            # to deepest, which keeps 0 m visible for the same reason.
            top_req = depth_min if depth_min is not None else 0.0
            bot_req = depth_max if depth_max is not None else yhi
            pad = (bot_req - top_req) * Y_PAD_FRAC or 1.0
            lo, hi = top_req - pad, bot_req + pad
        else:
            lo, hi = _padded(ylo, yhi, Y_PAD_FRAC)
        yrange = [hi, lo] if invert else [lo, hi]   # descending → down is deeper

        head = f"{y_channel['name']} vs {s['name']}"
        if LOCATION:
            head = f"{LOCATION}: {head}"

        fig = go.Figure(traces)
        fig.update_layout(
            title=dict(text=head, x=0.5, xanchor="center", font=dict(size=16)),
            # Axis on top: a profile is read downward, so the variable belongs at
            # the surface end where the reader starts.
            xaxis=dict(range=[x0, x1], side="top",
                       title=dict(text=s["label"], font=dict(size=AXIS_FONT), standoff=8)),
            yaxis=dict(range=yrange,
                       title=dict(text=ylabel, font=dict(size=AXIS_FONT), standoff=8)),
            template="plotly_white", hovermode="closest",
            width=760, height=620,
            legend=dict(title="Station"),
            margin=dict(l=70, r=30, t=110, b=40),
        )
        figs.append((s["name"], fig))
    return figs


def write_combined_html(figs, path, title="CTD Profiles", inline=False, show_heading=True):
    """All figures in one HTML file.

    inline=False links the plotting library from the web: ~30 KB, opens at once,
    emails and uploads without trouble, but needs a connection to draw.
    inline=True bakes the library in: several MB, works with no internet. Keep
    that one for presenting somewhere without wifi — a file that large is slow
    to open and easy to truncate in transit, so it is a poor default."""
    parts = [f.to_html(full_html=False,
                       include_plotlyjs=("inline" if inline else "cdn") if i == 0 else False)
             for i, (_, f) in enumerate(figs)]
    html = (
        '<!doctype html><html><head><meta charset="utf-8">'
        f"<title>{title}</title><style>"
        "body{font-family:system-ui,-apple-system,'Segoe UI',sans-serif;margin:24px;"
        "background:#fff;color:#111}h1{font-size:20px;font-weight:600}"
        ".grid{display:flex;flex-wrap:wrap;gap:16px}"
        # never let a figure be squeezed to nothing by the flex container
        ".grid>div{flex:0 0 auto}</style></head><body>"
        + (f"<h1>{title}</h1>" if show_heading else "")
        + f"<div class=\"grid\">{''.join(parts)}</div></body></html>"
    )
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)


def export_figures(figs, tag="", want_png=True):
    """Write the interactive HTML (+ optional PNGs) and return a status string."""
    os.makedirs(os.path.join(OUT_DIR, "png"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, "single_graphs"), exist_ok=True)
    base = f"{LOCATION} — CTD Profiles" if LOCATION else "CTD Profiles"
    html_path = os.path.join(OUT_DIR, "CTD_profiles.html")
    offline_path = os.path.join(OUT_DIR, "CTD_profiles_offline.html")
    write_combined_html(figs, html_path, title=f"{base}{tag}", inline=False)
    write_combined_html(figs, offline_path, title=f"{base}{tag}", inline=True)

    # One file per variable, so a single graph can be shared or embedded on its
    # own without anyone having to edit HTML. No heading: the figure carries its
    # own title, and a bare graph embeds more cleanly in someone else's page.
    for name, fig in figs:
        safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
        one = f"{LOCATION}: Depth vs {name}" if LOCATION else f"Depth vs {name}"
        write_combined_html([(name, fig)],
                            os.path.join(OUT_DIR, "single_graphs", f"{safe}.html"),
                            title=one, inline=False, show_heading=False)

    msg = (f"{html_path} ({os.path.getsize(html_path)/1e3:.0f} KB)"
           f" + offline copy ({os.path.getsize(offline_path)/1e6:.1f} MB)"
           f" + {len(figs)} single graphs")
    if want_png and EXPORT_PNG:
        try:
            for name, fig in figs:
                safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
                fig.write_image(os.path.join(OUT_DIR, "png", f"{safe}.png"), scale=3)
            msg += f" · {len(figs)} PNGs"
        except Exception as e:
            msg += f" · PNG skipped ({type(e).__name__})"
    return msg


print("Setup complete.  markers:", SHOW_MARKERS, "· x-padding:", f"{X_PAD_FRAC:.0%}")

In [ ]:
#@title 1 · Survey location and files
#@markdown **Survey location** — the name of the overall area you sampled.
#@markdown It becomes the title of every graph, for example
#@markdown `Quartermaster Harbor: Depth vs Temperature`. Leave it blank to get
#@markdown just `Depth vs Temperature`.
survey_location = "" #@param {type:"string"}
#@markdown ---
#@markdown **Keep the downcast only.** A cast that has not been loop edited still
#@markdown holds the upcast, so the profile doubles back on itself. Ticked, this
#@markdown cuts at the deepest reading and keeps readings deeper than everything
#@markdown before them. Files already loop edited are left untouched.
#@markdown
#@markdown This is a **display fix, not processing.** It makes the shape readable;
#@markdown it does not correct the values. Sea-Bird's pipeline also runs Filter,
#@markdown AlignCTD and CellTM *before* deriving salinity, and those need
#@markdown scan-rate data this notebook does not have. A truly raw file can still
#@markdown show salinity spikes at sharp gradients.
downcast_only_raw = True #@param {type:"boolean"}
#@markdown ---
#@markdown Run this cell and pick your `.cnv` files when the upload button appears.
#@markdown Picked the wrong file? Run this cell again and re-upload — only the
#@markdown newest copy of each station is used.

import ipywidgets as widgets
from IPython.display import display

LOCATION = survey_location.strip()
raw = load_files()

stations = []
# Sort on the cleaned station name, never the filename — a ' (1)' left by a
# re-upload would otherwise reorder stations and shuffle every colour.
for fn in sorted(raw, key=lambda f: natkey(station_label(f))):
    try:
        df, proc = parse_cnv(raw[fn], fn)
    except ValueError as e:
        print(f"  SKIPPED  {e}")
        continue
    if df.empty:
        print(f"  SKIPPED  {fn}: no data rows after *END*")
        continue

    label = station_label(fn)
    dcol, dlab = find_channel(df, DEPTH_CANDIDATES)

    note = ""
    if "loopedit" not in proc:
        if downcast_only_raw:
            df, dropped = downcast_only(df, dcol)
            # Not "loopedit" — that is a Sea-Bird step this notebook does not run.
            proc = proc | {"downcast cut (this notebook)"}
            note = f"  ← raw cast, kept the downcast ({dropped:,} rows dropped)"
        else:
            note = "  ← raw cast, may double back on itself"

    # Filter / AlignCTD / CellTM run before Derive and change the values, not just
    # the shape. Nothing here can substitute for them, so say so rather than let a
    # tidy-looking profile imply the numbers were corrected.
    missing = [s for s in ("align", "celltm") if not any(s in p for p in proc)]
    if missing and "derive" in proc:
        note += ("\n      NOTE  no alignment or cell thermal-mass correction in this "
                 "file's header — salinity and oxygen may be off at sharp gradients.")

    stations.append((label, df))
    drange = f"{df[dcol].min():.1f}–{df[dcol].max():.1f}" if dcol else "no depth column"
    print(f"  {label:<28} {len(df):>6,} rows   {drange:<16} "
          f"processing: {', '.join(sorted(proc)) or 'none'}{note}")

if not stations:
    raise SystemExit("No readable .cnv files found.")

DEEPEST = max(df[find_channel(df, DEPTH_CANDIDATES)[0]].max() for _, df in stations)
assign_station_styles([lab for lab, _ in stations])

print(f"\n{len(stations)} station(s) loaded · deepest reading {DEEPEST:.1f} m")
print(f"Location: {LOCATION or '(none set)'}")
print("\nColour key — reuse these in any other chart of the same stations:")
for lab, col in STATION_COLORS.items():
    print(f"  {col}   {lab}")


# ─────────── choose what to plot ───────────
RECOGNISED, EXTRAS, Y_DEFAULT = available_channels(stations)


def _row(chan, ticked):
    box = widgets.Checkbox(value=ticked, description=chan["name"], indent=False,
                           layout=widgets.Layout(width="250px"))
    txt = widgets.Text(value=chan["label"], layout=widgets.Layout(width="270px"))
    return box, txt, chan


# Recognised variables start ticked; every other column the files carry is
# offered unticked rather than hidden, so nothing is silently unavailable.
CHANNEL_ROWS = ([_row(c, True) for c in RECOGNISED]
                + [_row(c, False) for c in EXTRAS])

Y_CHOICES = {c["name"]: c for c in [Y_DEFAULT] + RECOGNISED + EXTRAS}
Y_PICK = widgets.Dropdown(options=list(Y_CHOICES), value=Y_DEFAULT["name"],
                          description="Y axis:", style={"description_width": "62px"},
                          layout=widgets.Layout(width="290px"))
Y_LABEL = widgets.Text(value=Y_DEFAULT["label"], description="Y label:",
                       style={"description_width": "62px"},
                       layout=widgets.Layout(width="290px"))
Y_INVERT = widgets.Checkbox(value=True, description="Invert Y axis (largest at the bottom)",
                            indent=False, layout=widgets.Layout(width="360px"))
Y_PICK.observe(lambda _c: setattr(Y_LABEL, "value", Y_CHOICES[Y_PICK.value]["label"]),
               names="value")


def selected_series():
    """Ticked channels, with whatever axis label sits in the box beside each."""
    return [{"name": ch["name"], "col": ch["col"], "label": txt.value}
            for box, txt, ch in CHANNEL_ROWS if box.value]


def selected_y():
    # is_depth stays a property of the channel — it governs the depth window and
    # surface anchoring. invert is purely the tick box, so unticking it flips the
    # axis without turning the depth window off.
    ch = Y_CHOICES[Y_PICK.value]
    return {"name": ch["name"], "col": ch["col"], "label": Y_LABEL.value,
            "is_depth": bool(ch.get("is_depth", False)),
            "invert": bool(Y_INVERT.value)}


display(widgets.HTML(
    "<b>Plot these</b> &mdash; tick what you want a graph of. The box beside each "
    "is its axis label, units included; edit it if you like.<br>"
    "Recognised variables are already ticked. The rest are other columns your "
    "files happen to contain, in case you want them."))
display(widgets.VBox([widgets.HBox([b, t]) for b, t, _ in CHANNEL_ROWS]))
display(widgets.HTML(
    "<b>Y axis</b> &mdash; depth unless you change it. Pick something else and the "
    "graphs become ordinary scatter plots titled <i>Salinity vs Temperature</i> "
    "and so on, and the depth window in the next cell no longer applies."))
display(widgets.HBox([Y_PICK, Y_LABEL]))
display(Y_INVERT)
print("\nNow run the Plot cell.")

In [ ]:
#@title 2 · Draw the graphs
#@markdown **Which part of the water column to show**, in metres below the surface.
#@markdown Leave both blank to show the whole cast, top to bottom.
#@markdown
#@markdown To look at just the surface layer, put `0` and `20` — that shows
#@markdown everything between 0 and 20 metres deep. Change the numbers and run
#@markdown this cell again; your files stay loaded. (Ignored if you moved the
#@markdown Y axis off depth.)
depth_from_m = "" #@param {type:"string"}
depth_to_m   = "" #@param {type:"string"}

try:
    stations
except NameError:
    raise SystemExit("Run the Load cell first.")


def _num(s):
    s = str(s).strip()
    if not s:
        return None
    try:
        return float(s)
    except ValueError:
        print(f"  ignoring '{s}' — that is not a number")
        return None


y = selected_y()
series = [s for s in selected_series() if s["col"].lower() != y["col"].lower()]

if not series:
    raise SystemExit("Nothing ticked to plot. Tick at least one variable in the Load cell.")

dmin, dmax = _num(depth_from_m), _num(depth_to_m)
if dmin is not None and dmax is not None and dmin > dmax:
    dmin, dmax = dmax, dmin

figs = build_figures(stations, series, y, dmin, dmax)

if not figs:
    where = f" between {dmin:g} and {dmax:g} m" if (dmin or dmax) else ""
    print(f"Nothing to draw{where}. The deepest reading in this set is {DEEPEST:.1f} m.")
else:
    span = ""
    if y["is_depth"]:
        lo = f"{dmin:g}" if dmin is not None else "0"
        hi = f"{dmax:g}" if dmax is not None else f"{DEEPEST:.0f}"
        span = f" · {lo}–{hi} m"
    print(f"{y['name']} vs: " + ", ".join(n for n, _ in figs) + span)
    print("Saved:", export_figures(figs, tag=span))
    # Plain fig.show() with no clear_output(). Colab's plotly renderer attaches a
    # MutationObserver that purges the plot when its output element is rebuilt, so
    # clearing and redrawing the cell destroys the figures as they arrive.
    for _, f in figs:
        f.show()